# 06_modelo_baseline_ml_formal

## Objetivo
Construir un baseline supervisado transparente para hostilidad usando el corpus formal
`media_anchored` y la muestra manual enriquecida por lexicón.

## Contrato manual consumido
- `hostility_relevance`: nivel `0`, `1`, `2` o `3`.
- `manual_hostility`: `0` no se encontró hostilidad/incivilidad; `1` sí se encontró.
- `manual_hate_speech`: `0` no se encontró odio; `1` sí se encontró.
- `notes`: contexto o justificación opcional.

El baseline principal usa `manual_hostility` normalizada como target binario. La
variable `manual_hate_speech` se conserva como un segundo target binario distinto y el
nivel `0–3` como target multiclase. Las decisiones humanas nunca se sobrescriben.


## Advertencias metodológicas
1. Este notebook construye un **baseline**, no un modelo definitivo.
2. La calidad del modelo depende de la calidad y cantidad de etiquetas manuales.
3. Un lexicón no equivale a detección definitiva de discurso de odio.
4. Hostilidad no es lo mismo que discurso de odio.
5. Puede existir `manual_hostility=1` con `manual_hate_speech=0`.
6. Con pocas etiquetas, las métricas son inestables.
7. El corpus está anclado en medios costarricenses; no representa todo X.
8. El modelo puede aprender sesgos de medios, ventana temporal o esquema de etiquetado.
9. Para el TFM, este baseline sirve como punto de comparación y diagnóstico.
10. La muestra formal está enriquecida por lexicón; las métricas no estiman prevalencia poblacional.
11. `src/labels.py` valida las tres etiquetas, pero no reemplaza los valores humanos.
12. Los niveles son categorías jerárquicas y no deben interpretarse como una intensidad continua pura.


## A) Setup


In [ ]:
import os
import sys
import importlib
import re
import json
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import joblib
    JOBLIB_AVAILABLE = True
except Exception:
    JOBLIB_AVAILABLE = False
    joblib = None
    print("[WARN] joblib no disponible. Se omitirá persistencia del modelo en .joblib.")

SKLEARN_AVAILABLE = True
SKLEARN_IMPORT_ERROR = None
try:
    from sklearn.dummy import DummyClassifier
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
    )
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.svm import LinearSVC
except Exception as exc:
    SKLEARN_AVAILABLE = False
    SKLEARN_IMPORT_ERROR = str(exc)
    DummyClassifier = None
    TfidfVectorizer = None
    LogisticRegression = None
    accuracy_score = None
    classification_report = None
    confusion_matrix = None
    f1_score = None
    precision_score = None
    recall_score = None
    train_test_split = None
    Pipeline = None
    LinearSVC = None
    print(f"[WARN] sklearn no disponible: {exc}")


# Reproducibilidad
RANDOM_STATE = 42
TEST_SIZE = 0.25
MIN_LABELED_ROWS = int(os.getenv("MIN_LABELED_ROWS", "50"))
MIN_POSITIVE_ROWS = int(os.getenv("MIN_POSITIVE_ROWS", "10"))
MIN_NEGATIVE_ROWS = int(os.getenv("MIN_NEGATIVE_ROWS", "10"))
LABELING_SAMPLE_SIZE = int(os.getenv("LABELING_SAMPLE_SIZE", "300"))

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True


In [ ]:
def is_project_dir(path):
    return (path / "config").exists() and (path / "data").exists() and (path / "notebooks").exists()


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if is_project_dir(candidate):
            return candidate
        child = candidate / "HateCR"
        if is_project_dir(child):
            return child
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.labels as label_utils
importlib.reload(label_utils)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS_ROOT = PROJECT_ROOT / "reports"
FORMAL_EDA_REPORTS_DIR = REPORTS_ROOT / "formal_eda"
REPORTS_DIR = REPORTS_ROOT / "formal_ml"
MODELS_DIR = PROJECT_ROOT / "models" / "formal"
FIGURES_DIR = REPORTS_DIR / "figures"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("LABEL_SOURCE_DIR:", FORMAL_EDA_REPORTS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("labels module:", label_utils.__file__)

## B) Carga segura de datos


In [ ]:
def safe_read_csv(path, name):
    if not path.exists():
        print(f"[WARN] Falta {name}: {path}")
        return pd.DataFrame()
    decode_errors = []
    for encoding in ["utf-8-sig", "utf-8", "latin-1"]:
        try:
            df = pd.read_csv(path, encoding=encoding)
            for id_column in ["tweet_id", "reply_id", "source_post_id"]:
                if id_column in df.columns:
                    df[id_column] = df[id_column].astype("string")
            print(f"[OK] {name}: {len(df):,} filas (encoding={encoding})")
            return df
        except UnicodeDecodeError as exc:
            decode_errors.append(f"{encoding}: {exc}")
        except Exception as exc:
            print(f"[WARN] Error cargando {name}: {exc}")
            return pd.DataFrame()
    print(f"[WARN] No se pudo decodificar {name}: {' | '.join(decode_errors)}")
    return pd.DataFrame()


def ensure_col(df, col, default=np.nan):
    if col not in df.columns:
        df[col] = default
    return df


def normalize_text(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    value = str(text).lower().strip()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    value = re.sub(r"@\w+", " ", value)
    value = re.sub(r"#(\w+)", r"\1", value)
    value = unicodedata.normalize("NFD", value)
    value = "".join(ch for ch in value if unicodedata.category(ch) != "Mn")
    value = re.sub(r"\s+", " ", value).strip()
    return value


def pick_first_existing(paths):
    for name, path in paths:
        df = safe_read_csv(path, name)
        if not df.empty:
            return df, name, path
    return pd.DataFrame(), None, None


In [ ]:
# Corpus formal principal. EDA se prioriza porque ya contiene indicadores del lexicón.
ALLOW_LEGACY_CORPUS = os.getenv("ALLOW_LEGACY_CORPUS", "false").strip().lower() == "true"
corpus_candidates = [
    (
        "formal_eda",
        DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_eda.csv",
    ),
    (
        "formal_training_dedup",
        DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_training_dedup.csv",
    ),
    (
        "formal_analysis",
        DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_analysis.csv",
    ),
]
if ALLOW_LEGACY_CORPUS:
    corpus_candidates.extend([
        ("legacy_eda_dedup", DATA_PROCESSED / "x_media_anchored_interactions_corpus_eda_dedup.csv"),
        ("legacy_eda", DATA_PROCESSED / "x_media_anchored_interactions_corpus_eda.csv"),
    ])

corpus_df, corpus_source_name, corpus_source_path = pick_first_existing(corpus_candidates)
if corpus_df.empty:
    raise FileNotFoundError(
        "No se encontró corpus formal. Ejecuta primero los notebooks 03 y 04."
    )

REMOVE_DUPLICATE_COMMENTS_QUOTES = os.getenv(
    "REMOVE_DUPLICATE_COMMENTS_QUOTES", "true"
).strip().lower() == "true"
DEDUP_DROP_TEXT_DUPLICATES = os.getenv(
    "DEDUP_DROP_TEXT_DUPLICATES", "true"
).strip().lower() == "true"
DEDUP_TARGET_SOURCE_TYPES = {"reply_to_media_post", "quote_of_media_post"}

for column in ["source_type", "tweet_id", "text", "text_norm", "text_norm_hash", "created_at"]:
    corpus_df = ensure_col(corpus_df, column, np.nan)
if corpus_df["text_norm"].isna().any():
    base_text = corpus_df["text_norm"].where(corpus_df["text_norm"].notna(), corpus_df["text"])
    corpus_df["text_norm"] = base_text.fillna("").astype(str).map(normalize_text)
if corpus_df["text_norm_hash"].isna().all():
    corpus_df["text_norm_hash"] = corpus_df["text_norm"]

rows_before = len(corpus_df)
target_mask = corpus_df["source_type"].isin(DEDUP_TARGET_SOURCE_TYPES)
target_df = corpus_df[target_mask].copy()
other_df = corpus_df[~target_mask].copy()
before_target = len(target_df)

if REMOVE_DUPLICATE_COMMENTS_QUOTES:
    target_df = target_df.drop_duplicates(subset=["source_type", "tweet_id"], keep="first")
after_tweetid = len(target_df)
if REMOVE_DUPLICATE_COMMENTS_QUOTES and DEDUP_DROP_TEXT_DUPLICATES:
    non_empty = target_df["text_norm_hash"].fillna("").astype(str).str.strip().ne("")
    target_df = pd.concat([
        target_df[non_empty].drop_duplicates(subset=["source_type", "text_norm_hash"], keep="first"),
        target_df[~non_empty],
    ], ignore_index=False, sort=False)
after_text = len(target_df)
corpus_df = pd.concat([target_df, other_df], ignore_index=True, sort=False)

rows_after = len(corpus_df)
dedup_summary = pd.DataFrame([{
    "rows_before": rows_before,
    "rows_after": rows_after,
    "removed_total": rows_before - rows_after,
    "target_rows_before": before_target,
    "target_rows_after_tweetid": after_tweetid,
    "target_rows_after_text": after_text,
    "source_corpus_loaded": corpus_source_name,
}])

labeling_cols = [
    "source_type", "anchor_post_id", "anchor_media_id", "anchor_media_handle",
    "event_id", "event_name", "tweet_id", "text", "text_norm", "text_norm_hash",
    "created_at", "reply_author_id_hash", "conversation_id", "lang",
    "reply_count", "quote_count", "retweet_count", "like_count",
    "lexicon_hit_count", "categorized_lexicon_hit_count",
    "has_lexicon_match", "has_categorized_lexicon_match",
    "lexicon_terms_found", "lexicon_categories_found",
]
for column in label_utils.HUMAN_LABEL_COLUMNS:
    if column not in corpus_df.columns:
        corpus_df[column] = pd.NA
existing_cols = [column for column in labeling_cols if column in corpus_df.columns]
corpus_df = corpus_df[existing_cols + label_utils.HUMAN_LABEL_COLUMNS].copy()

dedup_out_path = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_model_input.csv"
dedup_summary_path = REPORTS_DIR / "corpus_training_dedup_summary.csv"
corpus_df.to_csv(dedup_out_path, index=False)
dedup_summary.to_csv(dedup_summary_path, index=False)

print("Corpus formal usado:", corpus_source_name)
print("Filas antes:", rows_before)
print("Filas tras deduplicación:", rows_after)
print("[OK]", dedup_out_path)
display(dedup_summary)

In [ ]:
# La muestra formal enriquecida por lexicón es la fuente canónica.
ALLOW_LEGACY_LABELS = os.getenv("ALLOW_LEGACY_LABELS", "false").strip().lower() == "true"
label_candidates = [
    (
        "formal_manual_review_sample_labeled",
        FORMAL_EDA_REPORTS_DIR / "manual_review_sample_labeled.csv",
    ),
    (
        "formal_manual_review_sample",
        FORMAL_EDA_REPORTS_DIR / "manual_review_sample.csv",
    ),
]
if ALLOW_LEGACY_LABELS:
    label_candidates.extend([
        ("legacy_manual_review_sample_labeled", REPORTS_ROOT / "manual_review_sample_labeled.csv"),
        ("legacy_manual_review_labeled", DATA_PROCESSED / "manual_review_labeled.csv"),
    ])

label_raw_candidates = []
for name, candidate_path in label_candidates:
    frame = safe_read_csv(candidate_path, name)
    if not frame.empty:
        label_raw_candidates.append((name, candidate_path, frame))

print("Candidatos formales de etiquetas cargados:", len(label_raw_candidates))
print("ALLOW_LEGACY_LABELS:", ALLOW_LEGACY_LABELS)

## C) Validación de etiquetas


In [ ]:
selected_label_df = pd.DataFrame()
selected_label_source = None
selected_label_path = None
selected_target_col = "y_hostility (normalización de manual_hostility)"
selected_label_diagnostics = None

for name, candidate_path, frame in label_raw_candidates:
    prepared, diagnostics = label_utils.prepare_manual_annotations(frame, strict=False)
    valid_n = int(prepared["annotation_ready_for_training"].sum())
    any_labeled_n = int(
        diagnostics["summary"].loc[
            diagnostics["summary"]["metric"].eq("rows_with_any_label"), "value"
        ].iloc[0]
    )
    print(
        f"[{name}] filas_con_alguna_etiqueta={any_labeled_n} "
        f"filas_completas_validas={valid_n}"
    )
    # Keep diagnostics for the canonical formal sample even while incomplete.
    if selected_label_diagnostics is None:
        selected_label_diagnostics = diagnostics
    if valid_n > 0:
        selected_label_df = prepared
        selected_label_source = name
        selected_label_path = candidate_path
        selected_label_diagnostics = diagnostics
        break

if selected_label_diagnostics is not None:
    for diagnostic_name, diagnostic_df in selected_label_diagnostics.items():
        diagnostic_df.to_csv(
            REPORTS_DIR / f"annotation_{diagnostic_name}.csv", index=False
        )

print("Fuente de etiquetas seleccionada:", selected_label_source)
print("Target principal:", selected_target_col)
print("Contrato: nivel 0-3, manual_hostility 0/1 y manual_hate_speech 0/1")

In [ ]:
# Preparar dataset etiquetado para modelado sin alterar valores humanos.
corpus_work = corpus_df.copy()
for column in [
    "tweet_id", "text", "text_norm", "source_type", "anchor_media_handle",
    "event_id", "created_at", "categorized_lexicon_hit_count",
    "lexicon_terms_found", "lexicon_categories_found",
]:
    corpus_work = ensure_col(corpus_work, column, np.nan)

corpus_merge = corpus_work.drop_duplicates(subset=["tweet_id"], keep="first").copy()
labeled_merged = pd.DataFrame()

if selected_label_df.empty:
    print("[WARN] La muestra formal aún no contiene etiquetas completas y válidas.")
else:
    label_df = selected_label_df.copy()
    valid_labels_df = label_df[label_df["annotation_ready_for_training"]].copy()
    print("Filas manuales completas y válidas:", len(valid_labels_df))

    corpus_columns = [
        "tweet_id", "text", "text_norm", "source_type", "anchor_media_handle",
        "event_id", "created_at", "categorized_lexicon_hit_count",
        "lexicon_terms_found", "lexicon_categories_found",
    ]
    labeled_merged = valid_labels_df.merge(
        corpus_merge[corpus_columns],
        on="tweet_id",
        how="left",
        suffixes=("_label", ""),
    )
    if "text_label" in labeled_merged.columns:
        labeled_merged["text"] = labeled_merged["text"].fillna(
            labeled_merged["text_label"]
        )

for column in [
    "tweet_id", "text", "source_type", "anchor_media_handle", "event_id",
    "created_at", "categorized_lexicon_hit_count", "lexicon_terms_found",
    "lexicon_categories_found", "hostility_level_label", "y_hostility",
    "y_severe_or_identity", "y_hate_speech", "y_hostility_multiclass",
]:
    labeled_merged = ensure_col(labeled_merged, column, np.nan)

print("labeled_merged filas:", len(labeled_merged))

In [ ]:
# Diagnóstico de suficiencia de los dos targets binarios y del nivel multiclase.
label_stats = {
    "total_labeled_rows": int(labeled_merged["y_hostility"].notna().sum()),
    "n_hostility_positive": int((labeled_merged["y_hostility"] == 1).sum()),
    "n_hostility_negative": int((labeled_merged["y_hostility"] == 0).sum()),
    "n_hate_positive": int((labeled_merged["y_hate_speech"] == 1).sum()),
    "n_hate_negative": int((labeled_merged["y_hate_speech"] == 0).sum()),
    "n_level_2_or_3": int((labeled_merged["y_severe_or_identity"] == 1).sum()),
}

n_labeled = label_stats["total_labeled_rows"]
n_pos = label_stats["n_hostility_positive"]
n_neg = label_stats["n_hostility_negative"]
pos_pct = round((n_pos / n_labeled * 100), 2) if n_labeled else 0.0

print("Filas con las tres etiquetas completas y válidas:", n_labeled)
print("Hostilidad encontrada (manual_hostility=1):", n_pos)
print("Hostilidad no encontrada (manual_hostility=0):", n_neg)
print("% con hostilidad:", pos_pct)
print("Odio encontrado (manual_hate_speech=1):", label_stats["n_hate_positive"])
print("Odio no encontrado (manual_hate_speech=0):", label_stats["n_hate_negative"])
print("Niveles 2 o 3:", label_stats["n_level_2_or_3"])

if not labeled_merged.empty:
    print("Distribución de hostility_relevance:")
    display(labeled_merged["y_hostility_multiclass"].value_counts().sort_index())
    print("Cruce manual_hostility × manual_hate_speech:")
    display(pd.crosstab(labeled_merged["y_hostility"], labeled_merged["y_hate_speech"]))

if n_labeled < MIN_LABELED_ROWS:
    print(f"[WARN] Menos de {MIN_LABELED_ROWS} ejemplos etiquetados.")
if n_pos < MIN_POSITIVE_ROWS:
    print(f"[WARN] Menos de {MIN_POSITIVE_ROWS} ejemplos positivos de hostilidad.")
if n_neg < MIN_NEGATIVE_ROWS:
    print(f"[WARN] Menos de {MIN_NEGATIVE_ROWS} ejemplos negativos de hostilidad.")
if not SKLEARN_AVAILABLE:
    print(f"[WARN] sklearn no disponible ({SKLEARN_IMPORT_ERROR}).")

CAN_TRAIN = (
    n_labeled >= MIN_LABELED_ROWS
    and n_pos >= MIN_POSITIVE_ROWS
    and n_neg >= MIN_NEGATIVE_ROWS
    and SKLEARN_AVAILABLE
)
print("CAN_TRAIN baseline de hostilidad:", CAN_TRAIN)


## D) Preparar plantilla si no hay etiquetas suficientes


In [ ]:
canonical_sample_path = FORMAL_EDA_REPORTS_DIR / "manual_review_sample.csv"
if not CAN_TRAIN:
    if canonical_sample_path.exists():
        print("[INFO] Continúa el etiquetado en la muestra formal canónica:")
        print(canonical_sample_path)
        print("[INFO] No se crea una plantilla duplicada ni se sobrescribe la muestra.")
    else:
        print("[WARN] No existe la muestra formal. Ejecuta primero el notebook 04.")
    print("[INFO] Se omite entrenamiento hasta contar con etiquetas suficientes y válidas.")

## E) Limpieza de texto para modelado


In [ ]:
model_df = labeled_merged.copy()

if CAN_TRAIN:
    # Texto base: text_norm si existe; fallback a text
    model_df = ensure_col(model_df, "text_norm", np.nan)
    model_df = ensure_col(model_df, "text", "")

    text_base = model_df["text_norm"].where(model_df["text_norm"].notna(), model_df["text"])
    model_df["text_model"] = text_base.fillna("").astype(str).map(normalize_text)

    # Eliminar filas sin texto útil
    model_df = model_df[model_df["text_model"].str.len() > 0].copy()

    # Target final
    model_df["y_hostility"] = pd.to_numeric(model_df["y_hostility"], errors="coerce")
    model_df = model_df[model_df["y_hostility"].isin([0, 1])].copy()
    model_df["y_hostility"] = model_df["y_hostility"].astype(int)

    print("Filas para modelado despues de limpieza:", len(model_df))
    display(model_df[["tweet_id", "text_model", "y_hostility"]].head(5))
else:
    print("[SKIP] Sin etiquetas suficientes: no se prepara entrenamiento.")


## F) División entrenamiento/prueba


In [ ]:
split_info = {}

if CAN_TRAIN:
    y = model_df["y_hostility"].copy()

    try:
        train_df, test_df = train_test_split(
            model_df,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y,
        )
        split_info["used_stratify"] = True
    except Exception as exc:
        print(f"[WARN] train_test_split con stratify falló: {exc}")
        train_df, test_df = train_test_split(
            model_df,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=None,
        )
        split_info["used_stratify"] = False

    print("Train size:", len(train_df), "Test size:", len(test_df), "used_stratify:", split_info.get("used_stratify"))

    X_train = train_df["text_model"].values
    y_train = train_df["y_hostility"].values
    X_test = test_df["text_model"].values
    y_test = test_df["y_hostility"].values
else:
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()
    X_train = np.array([])
    y_train = np.array([])
    X_test = np.array([])
    y_test = np.array([])
    print("[SKIP] Sin entrenamiento.")


## G) Baselines requeridos
1. Dummy most_frequent
2. Lexicón (`hostility_hits` > 0)
3. TF-IDF + LogisticRegression
4. (Opcional) TF-IDF + LinearSVC


In [ ]:
def build_basic_hostility_hits(series_text):
    basic_lexicon = [
        "corrupto", "corrupta", "ladron", "ladrona", "mentiroso", "mentirosa",
        "vendido", "vendida", "sinverguenza", "payaso", "payasa", "dictador",
        "dictadora", "narco", "rata", "traidor", "traidora", "idiota", "imbecil", "basura"
    ]
    lex = sorted({normalize_text(t) for t in basic_lexicon if normalize_text(t)})

    def count_hits(txt):
        text = str(txt or "")
        hits = 0
        for term in lex:
            if re.search(rf"\b{re.escape(term)}\b", text):
                hits += 1
        return hits

    return series_text.map(count_hits)


def evaluate_predictions(model_name, y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    metrics_row = {
        "model": model_name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_hostile": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall_hostile": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_hostile": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "n_test": int(len(y_true)),
    }

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return metrics_row, report, cm


metrics_rows = []
reports_by_model = {}
cm_by_model = {}
preds_by_model = {}
score_by_model = {}

model_objects = {}
best_model_name = None
best_model_object = None

if CAN_TRAIN and len(train_df) > 0 and len(test_df) > 0:
    # 1) Dummy baseline
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(np.zeros((len(train_df), 1)), y_train)
    y_pred_dummy = dummy.predict(np.zeros((len(test_df), 1)))
    row, rep, cm = evaluate_predictions("dummy_most_frequent", y_test, y_pred_dummy)
    metrics_rows.append(row)
    reports_by_model["dummy_most_frequent"] = rep
    cm_by_model["dummy_most_frequent"] = cm
    preds_by_model["dummy_most_frequent"] = y_pred_dummy
    model_objects["dummy_most_frequent"] = dummy

    # 2) Lexicon baseline: use the same categorized lexicon filter as the sample.
    if (
        "categorized_lexicon_hit_count" in test_df.columns
        and test_df["categorized_lexicon_hit_count"].notna().any()
    ):
        test_hits = pd.to_numeric(
            test_df["categorized_lexicon_hit_count"], errors="coerce"
        ).fillna(0).astype(int)
        train_hits = pd.to_numeric(
            train_df["categorized_lexicon_hit_count"], errors="coerce"
        ).fillna(0).astype(int)
    else:
        print("[INFO] Indicador formal del lexicón no disponible; usando fallback básico.")
        train_hits = build_basic_hostility_hits(train_df["text_model"])
        test_hits = build_basic_hostility_hits(test_df["text_model"])

    y_pred_lex = (test_hits > 0).astype(int).values
    row, rep, cm = evaluate_predictions("lexicon_hits", y_test, y_pred_lex)
    metrics_rows.append(row)
    reports_by_model["lexicon_hits"] = rep
    cm_by_model["lexicon_hits"] = cm
    preds_by_model["lexicon_hits"] = y_pred_lex

    # 3) Logistic Regression + TFIDF
    logreg_pipe = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ])
    logreg_pipe.fit(X_train, y_train)
    y_pred_logreg = logreg_pipe.predict(X_test)
    row, rep, cm = evaluate_predictions("logreg_tfidf", y_test, y_pred_logreg)
    metrics_rows.append(row)
    reports_by_model["logreg_tfidf"] = rep
    cm_by_model["logreg_tfidf"] = cm
    preds_by_model["logreg_tfidf"] = y_pred_logreg
    model_objects["logreg_tfidf"] = logreg_pipe

    # score/proba si disponible
    if hasattr(logreg_pipe, "predict_proba"):
        score_by_model["logreg_tfidf"] = logreg_pipe.predict_proba(X_test)[:, 1]

    # 4) LinearSVC + TFIDF (opcional)
    try:
        svc_pipe = Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    sublinear_tf=True,
                ),
            ),
            (
                "clf",
                LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
            ),
        ])
        svc_pipe.fit(X_train, y_train)
        y_pred_svc = svc_pipe.predict(X_test)
        row, rep, cm = evaluate_predictions("linearsvc_tfidf", y_test, y_pred_svc)
        metrics_rows.append(row)
        reports_by_model["linearsvc_tfidf"] = rep
        cm_by_model["linearsvc_tfidf"] = cm
        preds_by_model["linearsvc_tfidf"] = y_pred_svc
        model_objects["linearsvc_tfidf"] = svc_pipe

        # decision_function como score proxy
        if hasattr(svc_pipe, "decision_function"):
            score_by_model["linearsvc_tfidf"] = svc_pipe.decision_function(X_test)

    except Exception as exc:
        print(f"[WARN] LinearSVC no se entrenó: {exc}")

metrics_df = pd.DataFrame(metrics_rows)
if not metrics_df.empty:
    metrics_df = metrics_df.sort_values(["f1_hostile", "macro_f1", "recall_hostile"], ascending=False).reset_index(drop=True)

display(metrics_df)

## H) Métricas


In [ ]:
if CAN_TRAIN and not metrics_df.empty:
    metrics_path = REPORTS_DIR / "baseline_ml_metrics.csv"
    metrics_df.to_csv(metrics_path, index=False)
    print("[OK] Métricas guardadas:", metrics_path)

    # Mostrar classification report de cada modelo
    for model_name in metrics_df["model"].tolist():
        print("\n===", model_name, "===")
        rep = reports_by_model.get(model_name, {})
        rep_df = pd.DataFrame(rep).T
        display(rep_df)
else:
    print("[SKIP] No hay métricas porque no se entrenó.")


## I) Matrices de confusión


In [ ]:
def plot_confusion(cm, title, path):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["no_hostil", "hostil"])
    ax.set_yticklabels(["no_hostil", "hostil"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


if CAN_TRAIN and not metrics_df.empty:
    conf_paths = {
        "dummy_most_frequent": FIGURES_DIR / "confusion_dummy.png",
        "lexicon_hits": FIGURES_DIR / "confusion_lexicon.png",
        "logreg_tfidf": FIGURES_DIR / "confusion_logreg_tfidf.png",
        "linearsvc_tfidf": FIGURES_DIR / "confusion_linearsvc_tfidf.png",
    }

    for model_name, out_path in conf_paths.items():
        if model_name in cm_by_model:
            plot_confusion(cm_by_model[model_name], f"Confusion Matrix - {model_name}", out_path)
            print("[OK]", out_path)
else:
    print("[SKIP] Sin matrices de confusión.")


## J) Interpretabilidad del modelo (Logistic Regression)


In [ ]:
logreg_features_df = pd.DataFrame()

if CAN_TRAIN and "logreg_tfidf" in model_objects:
    logreg_pipe = model_objects["logreg_tfidf"]
    tfidf = logreg_pipe.named_steps["tfidf"]
    clf = logreg_pipe.named_steps["clf"]

    features = tfidf.get_feature_names_out()
    coefs = clf.coef_[0]

    feat_df = pd.DataFrame({
        "feature": features,
        "coefficient": coefs,
    })
    feat_df["direction"] = np.where(feat_df["coefficient"] >= 0, "hostile", "non_hostile")

    top_hostile = feat_df.sort_values("coefficient", ascending=False).head(30)
    top_non_hostile = feat_df.sort_values("coefficient", ascending=True).head(30)

    logreg_features_df = pd.concat([top_hostile, top_non_hostile], ignore_index=True)
    features_path = REPORTS_DIR / "baseline_logreg_top_features.csv"
    logreg_features_df.to_csv(features_path, index=False)
    print("[OK] Guardado:", features_path)

    print("Top 30 hostiles")
    display(top_hostile)
    print("Top 30 no hostiles")
    display(top_non_hostile)

    # Gráfico top hostiles
    plt.figure(figsize=(12, 6))
    top_plot = top_hostile.sort_values("coefficient", ascending=True)
    plt.barh(top_plot["feature"], top_plot["coefficient"], color="#d62728")
    plt.title("Top términos asociados a hostilidad (LogReg)")
    plt.xlabel("Coeficiente")
    plt.tight_layout()
    top_fig_path = FIGURES_DIR / "logreg_top_hostile_features.png"
    plt.savefig(top_fig_path, dpi=150)
    plt.show()
    print("[OK] Guardado:", top_fig_path)
else:
    print("[SKIP] Logistic regression no disponible.")


## K) Análisis de errores (mejor baseline)


In [ ]:
best_model_name = None
best_model_object = None
pred_test_df = pd.DataFrame()
false_pos_df = pd.DataFrame()
false_neg_df = pd.DataFrame()

if CAN_TRAIN and not metrics_df.empty:
    supervised_metrics = metrics_df[
        metrics_df["model"].isin(["logreg_tfidf", "linearsvc_tfidf"])
    ].copy()
    if supervised_metrics.empty:
        print("[SKIP] No hay modelos supervisados disponibles para análisis de errores.")
    else:
        best_model_name = supervised_metrics.iloc[0]["model"]
        best_model_object = model_objects.get(best_model_name)
        print("Mejor modelo supervisado según f1_hostile:", best_model_name)

        y_pred_best = preds_by_model.get(best_model_name)

        pred_test_df = test_df.copy()
        pred_test_df["y_true"] = y_test
        pred_test_df["y_pred"] = y_pred_best
        pred_test_df["predicted_label"] = pred_test_df["y_pred"].map({1: "hostil", 0: "no_hostil"})

        if best_model_name in score_by_model:
            pred_test_df["model_score"] = score_by_model[best_model_name]
        else:
            pred_test_df["model_score"] = np.nan

        for c in [
            "tweet_id", "text", "source_type", "anchor_media_handle", "event_id",
            "created_at", "categorized_lexicon_hit_count", "lexicon_terms_found",
            "hostility_relevance_normalized", "hostility_level_label",
            "y_severe_or_identity", "y_hate_speech",
        ]:
            pred_test_df = ensure_col(pred_test_df, c, np.nan)

        keep_cols = [
            "tweet_id", "text", "y_true", "y_pred", "predicted_label", "source_type",
            "anchor_media_handle", "event_id", "created_at",
            "categorized_lexicon_hit_count", "lexicon_terms_found",
            "hostility_relevance_normalized", "hostility_level_label",
            "y_severe_or_identity", "y_hate_speech", "model_score",
        ]
        pred_test_df = pred_test_df[keep_cols].copy()

        false_pos_df = pred_test_df[(pred_test_df["y_true"] == 0) & (pred_test_df["y_pred"] == 1)].copy()
        false_neg_df = pred_test_df[(pred_test_df["y_true"] == 1) & (pred_test_df["y_pred"] == 0)].copy()

        preds_path = REPORTS_DIR / "baseline_predictions_test.csv"
        fp_path = REPORTS_DIR / "baseline_false_positives.csv"
        fn_path = REPORTS_DIR / "baseline_false_negatives.csv"

        pred_test_df.to_csv(preds_path, index=False)
        false_pos_df.to_csv(fp_path, index=False)
        false_neg_df.to_csv(fn_path, index=False)

        print("[OK]", preds_path)
        print("[OK]", fp_path)
        print("[OK]", fn_path)

        print("\n20 falsos positivos")
        display(false_pos_df.head(20))

        print("\n20 falsos negativos")
        display(false_neg_df.head(20))
else:
    print("[SKIP] Sin análisis de errores por falta de entrenamiento.")


## L) Entrenamiento final y guardado del modelo


In [ ]:
final_model = None
final_model_name = "logreg_tfidf"
label_info = {}

if CAN_TRAIN:
    # Logistic Regression is the deployable baseline because it is interpretable
    # and provides calibrated class probabilities for corpus ranking.
    final_model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ])

    final_model.fit(model_df["text_model"].values, model_df["y_hostility"].values)

    model_path = MODELS_DIR / "baseline_logreg_tfidf.joblib"
    if JOBLIB_AVAILABLE:
        joblib.dump(final_model, model_path)
        print("[OK] Modelo guardado:", model_path)
    else:
        print("[WARN] joblib no disponible: modelo entrenado pero no persistido en disco.")

    final_model_row = metrics_df[metrics_df["model"] == final_model_name]

    label_info = {
        "target_used": "y_hostility_from_manual_hostility",
        "label_source": str(selected_label_path) if selected_label_path else None,
        "manual_label_contract": {
            "human_fields": {
                "hostility_relevance": [0, 1, 2, 3],
                "manual_hostility": [0, 1],
                "manual_hate_speech": [0, 1],
            },
            "level_names": label_utils.LEVEL_NAMES,
            "primary_binary_target": "manual_hostility",
            "secondary_binary_target": "manual_hate_speech",
            "schema_version": "hatecr_4level_binary_v2",
        },
        "n_examples": int(len(model_df)),
        "class_distribution": {
            "0": int((model_df["y_hostility"] == 0).sum()),
            "1": int((model_df["y_hostility"] == 1).sum()),
        },
        "trained_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "model": "baseline_logreg_tfidf",
        "model_selection_note": (
            "LinearSVC is reported separately when it has better held-out metrics; "
            "LogisticRegression is persisted for interpretability and predict_proba."
        ),
        "best_supervised_model_test": best_model_name,
        "vectorizer": {
            "type": "TfidfVectorizer",
            "ngram_range": [1, 2],
            "min_df": 2,
            "max_df": 0.95,
            "sublinear_tf": True,
        },
        "main_metrics_test": (
            final_model_row.iloc[0].to_dict() if len(final_model_row) > 0 else {}
        ),
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "joblib_available": JOBLIB_AVAILABLE,
    }

    info_path = MODELS_DIR / "baseline_label_info.json"
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(label_info, f, ensure_ascii=False, indent=2)

    print("[OK] Metadata guardada:", info_path)
else:
    print("[SKIP] No se guardó modelo por falta de entrenamiento.")


## M) Aplicar modelo al corpus completo


In [ ]:
corpus_pred_df = corpus_work.copy()

if CAN_TRAIN and final_model is not None:
    corpus_pred_df = ensure_col(corpus_pred_df, "text_norm", np.nan)
    corpus_pred_df = ensure_col(corpus_pred_df, "text", "")

    text_base = corpus_pred_df["text_norm"].where(corpus_pred_df["text_norm"].notna(), corpus_pred_df["text"])
    corpus_pred_df["text_model"] = text_base.fillna("").astype(str).map(normalize_text)

    X_full = corpus_pred_df["text_model"].values

    corpus_pred_df["ml_hostility_pred"] = final_model.predict(X_full)
    corpus_pred_df["ml_model_name"] = final_model_name

    if hasattr(final_model, "predict_proba"):
        corpus_pred_df["ml_hostility_score"] = final_model.predict_proba(X_full)[:, 1]
    elif hasattr(final_model, "decision_function"):
        scores = final_model.decision_function(X_full)
        # escala simple 0-1 para ordenamiento aproximado
        min_s, max_s = np.min(scores), np.max(scores)
        if max_s > min_s:
            corpus_pred_df["ml_hostility_score"] = (scores - min_s) / (max_s - min_s)
        else:
            corpus_pred_df["ml_hostility_score"] = 0.5
    else:
        corpus_pred_df["ml_hostility_score"] = np.nan

    out_path = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_with_baseline_predictions.csv"
    corpus_pred_df.to_csv(out_path, index=False)
    print("[OK] Guardado:", out_path)
else:
    print("[SKIP] No se aplicó modelo al corpus completo.")

## N) Diagnósticos por medio y `source_type`


In [ ]:
if CAN_TRAIN and "ml_hostility_pred" in corpus_pred_df.columns:
    for c in ["source_type", "anchor_media_handle", "text", "event_id", "ml_hostility_score", "ml_hostility_pred"]:
        corpus_pred_df = ensure_col(corpus_pred_df, c, np.nan)

    by_source_type = (
        corpus_pred_df.groupby("source_type", dropna=False)
        .agg(
            total_rows=("ml_hostility_pred", "count"),
            hostile_pred_sum=("ml_hostility_pred", "sum"),
        )
        .reset_index()
    )
    by_source_type["hostile_pred_pct"] = (by_source_type["hostile_pred_sum"] / by_source_type["total_rows"] * 100).round(2)

    by_anchor_media = (
        corpus_pred_df.groupby("anchor_media_handle", dropna=False)
        .agg(
            total_rows=("ml_hostility_pred", "count"),
            hostile_pred_sum=("ml_hostility_pred", "sum"),
        )
        .reset_index()
        .sort_values("total_rows", ascending=False)
    )
    by_anchor_media["hostile_pred_pct"] = (by_anchor_media["hostile_pred_sum"] / by_anchor_media["total_rows"] * 100).round(2)

    top_hostile_texts = corpus_pred_df.sort_values("ml_hostility_score", ascending=False).head(200).copy()
    top_cols = [
        "tweet_id", "source_type", "anchor_media_handle", "event_id", "created_at", "text",
        "ml_hostility_pred", "ml_hostility_score"
    ]
    top_cols = [c for c in top_cols if c in top_hostile_texts.columns]
    top_hostile_texts = top_hostile_texts[top_cols]

    out1 = REPORTS_DIR / "predicted_hostility_by_source_type.csv"
    out2 = REPORTS_DIR / "predicted_hostility_by_anchor_media.csv"
    out3 = REPORTS_DIR / "top_predicted_hostile_texts.csv"

    by_source_type.to_csv(out1, index=False)
    by_anchor_media.to_csv(out2, index=False)
    top_hostile_texts.to_csv(out3, index=False)

    print("[OK]", out1)
    print("[OK]", out2)
    print("[OK]", out3)

    prediction_distribution = (
        corpus_pred_df["ml_hostility_pred"]
        .value_counts(dropna=False)
        .reindex([0, 1], fill_value=0)
        .rename_axis("prediction_value")
        .reset_index(name="n_comments")
    )
    prediction_distribution["prediction_label"] = prediction_distribution["prediction_value"].map(
        {0: "Predicción no hostil", 1: "Predicción hostil"}
    )
    prediction_distribution["percentage"] = (
        prediction_distribution["n_comments"] / prediction_distribution["n_comments"].sum() * 100
    ).round(2)

    distribution_out = REPORTS_DIR / "logreg_hostility_prediction_distribution.csv"
    prediction_distribution.to_csv(distribution_out, index=False)

    fig, ax = plt.subplots(figsize=(11, 3.4))
    colors = {0: "#355C68", 1: "#B24632"}
    left = 0.0
    for row in prediction_distribution.itertuples(index=False):
        ax.barh(
            ["Corpus completo"],
            [row.percentage],
            left=left,
            color=colors[int(row.prediction_value)],
            height=0.52,
            label=row.prediction_label,
        )
        count_label = f"{int(row.n_comments):,}".replace(",", ".")
        ax.text(
            left + row.percentage / 2,
            0,
            f"{row.prediction_label}\n{count_label} ({row.percentage:.2f}%)",
            ha="center",
            va="center",
            color="white",
            fontsize=11,
            fontweight="bold",
        )
        left += row.percentage

    total_label = f"{len(corpus_pred_df):,}".replace(",", ".")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Porcentaje de comentarios")
    ax.set_title(
        f"Logistic Regression: distribución exploratoria de predicciones de hostilidad\n"
        f"n = {total_label} comentarios"
    )
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    fig.text(
        0.5,
        0.025,
        "Predicciones del modelo; no representan todavía prevalencia real de hostilidad.",
        ha="center",
        fontsize=9,
        color="#555555",
    )
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    distribution_figure = FIGURES_DIR / "logreg_hostility_prediction_distribution.png"
    fig.savefig(distribution_figure, dpi=180, bbox_inches="tight")
    plt.show()

    print("[OK]", distribution_out)
    print("[OK]", distribution_figure)

    display(by_source_type.sort_values("hostile_pred_pct", ascending=False))
    display(by_anchor_media.head(20))
    display(top_hostile_texts.head(20))
else:
    print("[SKIP] Sin diagnósticos por falta de predicciones del modelo.")


## O) Conclusión operativa
Este notebook permite responder:
1. Si hay etiquetas suficientes para entrenar baseline.
2. Si TF-IDF + Logistic Regression supera a baseline trivial y lexicón.
3. Qué términos usa el modelo para hostilidad.
4. Qué errores comete (FP/FN).
5. Si conviene ampliar etiquetado antes de modelos más complejos.


## Salidas formales esperadas
Si hay etiquetas suficientes y válidas:
- `reports/formal_ml/baseline_ml_metrics.csv`
- `reports/formal_ml/baseline_logreg_top_features.csv`
- `reports/formal_ml/baseline_false_positives.csv`
- `reports/formal_ml/baseline_false_negatives.csv`
- `reports/formal_ml/baseline_predictions_test.csv`
- `models/formal/baseline_logreg_tfidf.joblib`
- `models/formal/baseline_label_info.json`
- `data/processed/x_media_anchored_interactions_corpus_formal_with_baseline_predictions.csv`

Mientras el etiquetado esté incompleto:
- diagnósticos `reports/formal_ml/annotation_*.csv`
- no se crea una plantilla duplicada ni se entrena el modelo.